# A/B Testing Lab

You're a data scientist at **WidgetCo**. Your PM keeps asking you to run experiments. Your job: design them, analyze the data, catch problems, make launch decisions.

**How it works:**
- Each lab gives you real-ish data and a task
- You write code to analyze it
- At the end, bring the notebook to Claude for feedback

**No reading. Just do.**

In [1]:
# Run this first
import sys
sys.path.insert(0, ".")
import numpy as np
import pandas as pd
from scipy import stats
from ab_lab import *
from ab_lab.data import generate_outlier_experiment, generate_experiment_with_covariate
from ab_lab.checker import check_srm
print("Lab ready.")

Lab ready.


In [2]:
lst = [5,1,2]

---
## Lab 1: Your First Experiment (~15 min)

**Situation:** PM wants to add a coupon code field to checkout. You need to:
1. **Design** the experiment (fill a config dict)
2. **Analyze** the data (compute means, difference, p-value)
3. **Decide**: launch or not?

In [3]:
# Step 1: DESIGN the experiment
# Fill in each field. Then run the cell to check your design.

my_design = {
    "hypothesis": "",          # what do you think will happen?
    "oec": "",                 # your main metric (one string)
    "guardrails": [],          # list of metrics that should NOT break
    "randomization_unit": "",  # "user", "session", or "page"?
    "population": "",          # who do you analyze?
    "min_duration_weeks": 0,   # how many weeks minimum?
}

check_design(my_design)

ISSUES FOUND:
  1. guardrails should be a list with at least 1 metric (e.g. latency, crash rate)
  2. min_duration_weeks should be >= 1 (need full week for day-of-week effect)


In [4]:
# Step 2: Here's your experiment data. Explore it.
scenario = get_scenario("coupon_field")
df = generate_experiment(
    n_control=scenario["n_per_group"],
    n_treatment=scenario["n_per_group"],
    control_mean=scenario["control_mean"],
    effect_pct=scenario["true_effect"],
    noise_std=scenario["noise_std"],
    seed=scenario["seed"],
)
print(f"Shape: {df.shape}")
df.head(10)

SCENARIO: Coupon Code Field on Checkout
  Your e-commerce site wants to add a coupon code field to checkout.
Concern: users might abandon checkout to hunt for codes.

  YOUR TASK: Design this experiment (fill in the design dict), then analyze the data.

Shape: (10000, 3)


,user_id,variant,revenue_per_user
0,6252,treatment,11.178513
1,4684,control,0.000000
2,1731,control,2.963340
3,4742,control,2.371707
4,4521,control,5.756874
5,6340,treatment,1.166625
6,576,control,13.899489
7,5202,treatment,0.000000
8,6363,treatment,3.397769
9,439,control,6.633124


In [ ]:
# Step 3: YOUR TURN — compute the basics
# Calculate:
#   1. Mean revenue_per_user for control and treatment
#   2. The difference (treatment - control)
#   3. The percent difference (diff / control_mean)

# YOUR CODE HERE
control = df[df["variant"] == "control"]["revenue_per_user"]
treatment = df[df["variant"] == "treatment"]["revenue_per_user"]

control_mean = ...   # TODO
treatment_mean = ... # TODO
diff = ...           # TODO
pct_diff = ...       # TODO

print(f"Control mean:    ${control_mean:.2f}")
print(f"Treatment mean:  ${treatment_mean:.2f}")
print(f"Difference:      ${diff:.2f} ({pct_diff:.2%})")

In [ ]:
# Step 4: YOUR TURN — is the difference real or noise?
# Run a two-sample t-test and get the p-value.
# Hint: stats.ttest_ind(group_a, group_b)

t_stat, p_value = ...  # TODO

print(f"t-statistic: {t_stat:.4f}")
print(f"p-value:     {p_value:.4f}")
print(f"Significant? {'YES' if p_value < 0.05 else 'NO'}")

In [ ]:
# Step 5: YOUR TURN — compute the 95% confidence interval
# CI = diff +/- 1.96 * standard_error
# standard_error = sqrt(var_control/n_control + var_treatment/n_treatment)

se = ...        # TODO: standard error
ci_lower = ...  # TODO
ci_upper = ...  # TODO

print(f"95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"Contains zero? {'YES — not significant' if ci_lower <= 0 <= ci_upper else 'NO — significant'}")

In [ ]:
# Step 6: DECISION TIME
# Based on your analysis, should WidgetCo add the coupon field?
# Run the checker to see if your reasoning holds up.

my_decision = ""  # "launch" or "don't launch" — fill this in

check_launch_decision(
    decision=my_decision,
    p_value=p_value,
    effect_pct=pct_diff,
    ci_lower=ci_lower / control_mean,  # as percent
    ci_upper=ci_upper / control_mean,
)

---
## Lab 2: Catch the Bug — Sample Ratio Mismatch (~10 min)

**Situation:** Your colleague ran an experiment and got a +5% lift. Looks great! But something smells off. The experiment was configured as 50/50 split.

**Your task:** Check if the split is actually 50/50. If not, the results are garbage.

In [ ]:
# Step 1: Load the suspicious experiment
df_srm = generate_srm_experiment()
print(f"Shape: {df_srm.shape}")
df_srm["variant"].value_counts()

In [ ]:
# Step 2: YOUR TURN — count users in each group
# Does this look like 50/50?

n_control = ...    # TODO: count control users
n_treatment = ...  # TODO: count treatment users

print(f"Control:   {n_control}")
print(f"Treatment: {n_treatment}")
print(f"Ratio:     {n_treatment / (n_control + n_treatment):.3f}")
print(f"\nDoes this look like 50/50 to you?")

In [ ]:
# Step 3: Run the formal SRM check
# This uses a chi-squared test to see if the split is significantly off

check_srm(n_control, n_treatment, expected_ratio=0.5)

# QUESTION: Based on this, should you trust the +5% lift?
# Write your answer as a comment below:
# MY ANSWER: 

---
## Lab 3: Outliers Are Eating Your Experiment (~10 min)

**Situation:** You ran an A/B test for a new feature. Treatment should have a +2% lift. But when you analyze the data, the result looks weird. A few users spent $50,000 each (bots? fraud?).

**Your task:** See how outliers break things, then fix it.

In [ ]:
# Step 1: Load data with outliers. Run t-test BEFORE fixing.
df_out = generate_outlier_experiment()

control_out = df_out[df_out["variant"] == "control"]["revenue_per_user"]
treatment_out = df_out[df_out["variant"] == "treatment"]["revenue_per_user"]

t_raw, p_raw = stats.ttest_ind(control_out, treatment_out)
print(f"=== BEFORE fixing outliers ===")
print(f"Control mean:   ${control_out.mean():.2f}")
print(f"Treatment mean: ${treatment_out.mean():.2f}")
print(f"Treatment max:  ${treatment_out.max():.0f}  <-- look at this!")
print(f"p-value: {p_raw:.4f}")
print(f"Significant? {'YES' if p_raw < 0.05 else 'NO'}")
print(f"\nThe true effect is +2%, but can your test detect it?")

In [ ]:
# Step 2: YOUR TURN — fix it by capping revenue at a reasonable max
# Hint: use .clip(upper=CAP_VALUE) on the revenue column

CAP_VALUE = ...  # TODO: pick a reasonable cap (e.g., 50? 100? 500?)

df_capped = df_out.copy()
df_capped["revenue_per_user"] = df_capped["revenue_per_user"].clip(upper=CAP_VALUE)

control_cap = df_capped[df_capped["variant"] == "control"]["revenue_per_user"]
treatment_cap = df_capped[df_capped["variant"] == "treatment"]["revenue_per_user"]

t_cap, p_cap = stats.ttest_ind(control_cap, treatment_cap)

print(f"=== AFTER capping at ${CAP_VALUE} ===")
print(f"Control mean:   ${control_cap.mean():.2f}")
print(f"Treatment mean: ${treatment_cap.mean():.2f}")
print(f"p-value: {p_cap:.4f}")
print(f"Significant? {'YES' if p_cap < 0.05 else 'NO'}")
print(f"\nCompare: p_before={p_raw:.4f} vs p_after={p_cap:.4f}")

---
## Lab 4: A/A Test — Is Your Platform Trustworthy? (~10 min)

**Situation:** Before running real experiments, you should check if your platform is healthy. Run an A/A test (both groups identical) many times. If more than ~5% show significant results, something is broken.

**Your task:** Simulate 200 A/A tests, compute false positive rate.

In [ ]:
# Step 1: YOUR TURN — simulate 200 A/A tests
# For each test:
#   1. Generate two groups from the SAME distribution (mean=3.2, std=8.0)
#   2. Run a t-test
#   3. Save the p-value
#
# Hint: use np.random.default_rng(seed) for each test

n_tests = 200
n_per_group = 1000
p_values = []

for i in range(n_tests):
    rng = np.random.default_rng(i)
    group_a = rng.normal(3.2, 8.0, n_per_group)
    group_b = rng.normal(3.2, 8.0, n_per_group)

    _, p = ...  # TODO: run t-test on group_a vs group_b
    p_values.append(p)

print(f"Collected {len(p_values)} p-values")

In [ ]:
# Step 2: YOUR TURN — what fraction of A/A tests are "significant"?
# This is your false positive rate. It should be ~5%.

n_significant = ...  # TODO: count how many p-values are < 0.05
false_positive_rate = ...  # TODO: n_significant / n_tests

print(f"Significant at p<0.05: {n_significant}/{n_tests} = {false_positive_rate:.1%}")

# Now let the checker evaluate
check_aa_analysis(p_values)

---
## Lab 5: CUPED — Make Your Experiment More Sensitive (~15 min)

**Situation:** You're testing a small change (+1% expected effect). With raw data, you can't detect it — p-value is too high. But you have **pre-experiment data** (what each user spent last week).

**Your task:** Use CUPED to reduce variance and detect the effect.

**What is CUPED?** Subtract out the predictable part of each user's metric using their pre-experiment behavior. Less noise → easier to detect small effects.

In [ ]:
# Step 1: Load experiment with pre-experiment covariate
df_cuped = generate_experiment_with_covariate()
print("Columns:", df_cuped.columns.tolist())
print(f"Shape: {df_cuped.shape}")
df_cuped.head()

In [ ]:
# Step 2: Try the naive approach — t-test on raw revenue_per_user
ctrl_raw = df_cuped[df_cuped["variant"] == "control"]["revenue_per_user"]
trt_raw = df_cuped[df_cuped["variant"] == "treatment"]["revenue_per_user"]

t_raw, p_raw = stats.ttest_ind(ctrl_raw, trt_raw)
print(f"=== WITHOUT CUPED ===")
print(f"Difference: {trt_raw.mean() - ctrl_raw.mean():.4f}")
print(f"p-value:    {p_raw:.4f}")
print(f"Can you detect the 1% effect? {'YES' if p_raw < 0.05 else 'NO — too noisy'}")

In [ ]:
# Step 3: YOUR TURN — apply CUPED
#
# CUPED formula:
#   Y_adjusted = Y - theta * (X - mean(X))
#   where theta = cov(Y, X) / var(X)
#   X = pre_experiment_revenue, Y = revenue_per_user
#
# This removes the predictable part of Y, reducing variance.

X = df_cuped["pre_experiment_revenue"]
Y = df_cuped["revenue_per_user"]

theta = ...  # TODO: np.cov(Y, X)[0, 1] / np.var(X)
df_cuped["revenue_cuped"] = ...  # TODO: Y - theta * (X - X.mean())

ctrl_cuped = df_cuped[df_cuped["variant"] == "control"]["revenue_cuped"]
trt_cuped = df_cuped[df_cuped["variant"] == "treatment"]["revenue_cuped"]

t_cuped, p_cuped = stats.ttest_ind(ctrl_cuped, trt_cuped)
print(f"=== WITH CUPED ===")
print(f"Difference: {trt_cuped.mean() - ctrl_cuped.mean():.4f}")
print(f"p-value:    {p_cuped:.4f}")
print(f"Can you detect the 1% effect? {'YES' if p_cuped < 0.05 else 'NO'}")
print(f"\nVariance reduction: {1 - ctrl_cuped.var()/ctrl_raw.var():.0%}")

---
## Lab 6: Power Analysis — How Many Users Do You Need? (~10 min)

**Situation:** PM asks "how long should we run this test?" You need to figure out the sample size.

**Your task:** Compute required sample size for different scenarios. See how effect size and variance change the answer.

In [ ]:
# Step 1: YOUR TURN — implement the sample size formula
#
# n ≈ 16 * σ² / δ²
#   where σ² = variance of the metric
#         δ  = minimum detectable effect (absolute)
#
# This gives n PER GROUP for 80% power at alpha=0.05

def sample_size_per_group(variance, min_effect):
    """Return required n per group for 80% power."""
    return ...  # TODO: implement the formula

# Test it: if variance=64, min_effect=0.5
n = sample_size_per_group(variance=64, min_effect=0.5)
print(f"Need {n:,.0f} users per group")

In [ ]:
# Step 2: See how effect size changes sample size dramatically
# Fill in the table — what happens when you want to detect smaller effects?

print("Effect size → Sample size (variance=64, per group)")
print("=" * 50)
for effect in [2.0, 1.0, 0.5, 0.1]:
    n = sample_size_per_group(64, effect)
    print(f"  Detect {effect:>5.1f} change → need {n:>10,.0f} users/group")

print("\nNotice: halving the effect size → 4x more users (quadratic!)")

In [ ]:
# Step 3: YOUR TURN — practical scenario
# WidgetCo gets 10,000 users/day. Revenue variance is 64.
# PM wants to detect a 1% change (baseline revenue = $3.20, so δ = 0.032)
# How many days do you need to run?

min_effect = ...  # TODO: 1% of $3.20
n_needed = sample_size_per_group(64, min_effect)
users_per_day = 10000
# each day gives 5000 per group (50/50 split)
days_needed = ...  # TODO: n_needed / (users_per_day / 2)

print(f"To detect a 1% change (${min_effect:.3f}):")
print(f"  Need {n_needed:,.0f} users per group")
print(f"  At {users_per_day:,} users/day → {days_needed:.0f} days")
print(f"\n  Practical? {'Yes' if days_needed <= 30 else 'Probably too long — consider CUPED or larger effect threshold'}")

---
## Lab 7: Multiple Scenarios — Design & Decide (~15 min)

**Situation:** Three new experiments land on your desk. Design each one and make a launch decision.

Pick at least 2. For each: design → analyze → decide.

In [ ]:
# See all available scenarios
list_scenarios()

In [ ]:
# YOUR TURN — pick a scenario, design it, analyze it
# Example with "new_ranking":

scenario = get_scenario("new_ranking")  # change this to try others

# 1. Design
my_design = {
    "hypothesis": "",
    "oec": "",
    "guardrails": [],
    "randomization_unit": "",
    "population": "",
    "min_duration_weeks": 0,
}
check_design(my_design)

In [ ]:
# 2. Analyze the data
metric = scenario.get("metric_name", "revenue_per_user")
df_scenario = generate_experiment(
    n_control=scenario["n_per_group"],
    n_treatment=scenario["n_per_group"],
    control_mean=scenario["control_mean"],
    effect_pct=scenario["true_effect"],
    noise_std=scenario["noise_std"],
    metric_name=metric,
    seed=scenario["seed"],
)

# YOUR TURN — compute means, run t-test, compute CI
ctrl = df_scenario[df_scenario["variant"] == "control"][metric]
trt = df_scenario[df_scenario["variant"] == "treatment"][metric]

# TODO: fill these in
ctrl_mean = ...
trt_mean = ...
diff = ...
pct_diff = ...
t_stat, p_val = ...
se = ...
ci_lo = ...
ci_hi = ...

print(f"Control:   {ctrl_mean:.4f}")
print(f"Treatment: {trt_mean:.4f}")
print(f"Diff:      {diff:.4f} ({pct_diff:.2%})")
print(f"p-value:   {p_val:.4f}")
print(f"95% CI:    [{ci_lo:.4f}, {ci_hi:.4f}]")

In [ ]:
# 3. DECIDE: launch or not?
my_decision = ""  # "launch" or "don't launch"

check_launch_decision(
    decision=my_decision,
    p_value=p_val,
    effect_pct=pct_diff,
    ci_lower=ci_lo / ctrl_mean,
    ci_upper=ci_hi / ctrl_mean,
)

---
## Done!

**Bring this notebook to Claude.** Say:
> "Review my A/B testing lab notebook"

Claude will look at your code, your decisions, and your designs — and give you targeted feedback on what you got right and what to review.

### What you practiced:
- **Lab 1:** Full experiment lifecycle (design → analyze → decide)
- **Lab 2:** Detecting Sample Ratio Mismatch
- **Lab 3:** Handling outliers
- **Lab 4:** A/A testing and false positive rates
- **Lab 5:** CUPED for variance reduction
- **Lab 6:** Power analysis and sample size
- **Lab 7:** Multiple scenarios end-to-end